# Step 5b Runner: continue UPR-CRE v0.1 mini-ablation with T4x2

This notebook is a **runner only**. It does not edit source code.

It does the following:

1. Pull the public repo/branch.
2. Prepare Kaggle RegDB layout.
3. Locate an existing `phase1_model_5.pth` from `/kaggle/input` or `/kaggle/working`.
4. Run two UPR-CRE configurations in parallel on T4x2:
   - GPU 0: `beta=0.1, gamma=0.0, warmup=2`
   - GPU 1: `beta=0.05, gamma=0.5, warmup=2`
5. Collect metrics and relation diagnostics.
6. Archive logs and summaries.
7. Optionally upload the final artifact to a GitHub Release.



## Block 0 — Configuration

Change only this block if needed. The default RegDB source matches your Kaggle dataset mount.

In [1]:
from pathlib import Path
import os
import time
from datetime import datetime

CFG = {
    # Public repo config
    "GITHUB_OWNER": "TranTruongMMCII",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",
    "WORK_DIR": "/kaggle/working",

    # Dataset config
    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",
    "REGDB_SOURCE": "/kaggle/input/datasets/xqq027/reg-db/RegDB",

    # Phase-1 checkpoint from Kaggle Input
    "PHASE1_CKPT_PATH": "/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth",

    # Run control
    "SEED": 1,
    "TRIAL": 1,
    "STAGE2_EPOCH": 15,
    "MILESTONES": "8 12",
    "BATCH_PIDNUM": 5,
    "PID_NUMSAMPLE": 4,
    "TEST_BATCH": 64,
    "NUM_WORKERS": 2,
    "LR": 0.00045,

    # Step 6: baseline vs best UPR v0.1
    "RUN_A_TAG": "baseline_p2s15",
    "RUN_A_ENABLE_UPR": False,
    "RUN_A_BETA": 0.0,
    "RUN_A_GAMMA": 0.0,
    "RUN_A_WARMUP": 0,

    "RUN_B_TAG": "upr_b01_g00_w2_p2s15",
    "RUN_B_ENABLE_UPR": True,
    "RUN_B_BETA": 0.1,
    "RUN_B_GAMMA": 0.0,
    "RUN_B_WARMUP": 2,

    # Reference rows from previous 5-epoch run.
    # Keep only for notes; do not compare directly with 15-epoch runs.
    "REFERENCE_BASELINE_R1": 0.4539,
    "REFERENCE_BASELINE_MAP": 0.4459,
    "REFERENCE_BASELINE_MINP": 0.3343,
    "REFERENCE_UPR_B01_G00_R1": 0.4913,
    "REFERENCE_UPR_B01_G00_MAP": 0.4707,
    "REFERENCE_UPR_B01_G00_MINP": 0.3640,

    # Do not use GitHub Release API for now.
    # Use the old git-push backup block after the run instead.
    "ENABLE_GITHUB_RELEASE_BACKUP": False,
    "GITHUB_TOKEN_SECRET": "GITHUB_TOKEN",
}

RUN_SUFFIX = datetime.utcnow().strftime("step6_%Y%m%d_%H%M%S")
print("RUN_SUFFIX =", RUN_SUFFIX)

repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"

print("repo_dir =", repo_dir)
print("wsl_dir =", wsl_dir)
print("checkpoint =", CFG["PHASE1_CKPT_PATH"])
print("checkpoint exists:", Path(CFG["PHASE1_CKPT_PATH"]).exists())

RUN_SUFFIX = step6_20260623_063755
repo_dir = /kaggle/working/UIT.CS2309
wsl_dir = /kaggle/working/UIT.CS2309/WSL_ReID
checkpoint = /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth
checkpoint exists: True


/tmp/ipykernel_58/1039276629.py:59: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_SUFFIX = datetime.utcnow().strftime("step6_%Y%m%d_%H%M%S")


## Block 1 — Pull repo and checkout branch

This block resets the Kaggle copy to `origin/feature/upr-cre`. It does not change code.

In [2]:
import subprocess
from pathlib import Path

repo_url = f"https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}.git"
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]

if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", CFG["BRANCH"], repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{CFG['BRANCH']}"], cwd=repo_dir, check=True)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir).decode().strip()
print("commit =", commit)
subprocess.run(["git", "status", "--short"], cwd=repo_dir, check=True)


Cloning into '/kaggle/working/UIT.CS2309'...


commit = 6c96804


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

## Block 2 — Install Kaggle requirements and prepare RegDB

This uses the scripts already committed in your repo. It also checks that two GPUs are visible.

In [3]:
import subprocess
from pathlib import Path

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"

subprocess.run(["pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=wsl_dir, check=True)
subprocess.run(["python", "scripts/apply_kaggle_compat_patches.py"], cwd=wsl_dir, check=True)

prepare_cmd = [
    "python", "scripts/prepare_regdb_kaggle.py",
    "--data-root", CFG["DATA_ROOT"],
]
if CFG["REGDB_SOURCE"]:
    prepare_cmd += ["--regdb-source", CFG["REGDB_SOURCE"]]
subprocess.run(prepare_cmd, cwd=wsl_dir, check=True)

subprocess.run([
    "python", "scripts/check_kaggle_env.py",
    "--data-root", CFG["DATA_ROOT"],
], cwd=wsl_dir, check=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
No compatibility patches were needed.
RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIREID_Dataset/RegDB/idx/train_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/train_thermal_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_thermal_1.txt OK

 train_visible_1.txt
first line: Visible/285/male_back_v_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Visible/285/male_back_v_05528_285.bmp
exists: True
image size: (64, 128) mode: RGB label: 0

 train_thermal_1.txt
first line: Thermal/285/male_back_t_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset

CompletedProcess(args=['python', 'scripts/check_kaggle_env.py', '--data-root', '/kaggle/working/VIREID_Dataset'], returncode=0)

## Block 3 — Static checks

This checks the code/scripts needed for Step 5. It does not check markdown docs.

In [4]:
from pathlib import Path
import subprocess

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
required = [
    "main.py",
    "wsl.py",
    "relation_metrics.py",
    "scripts/collect_relation_stats.py",
    "scripts/prepare_regdb_kaggle.py",
    "scripts/check_kaggle_env.py",
]
missing = [p for p in required if not (wsl_dir / p).exists()]
if missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing))

main_text = (wsl_dir / "main.py").read_text()
wsl_text = (wsl_dir / "wsl.py").read_text()
collector_text = (wsl_dir / "scripts/collect_relation_stats.py").read_text()

for marker in ["--upr-cre", "--upr-beta", "--upr-gamma", "--upr-warmup-epoch", "--save-relation-stats"]:
    assert marker in main_text, f"Missing {marker} in main.py"
for marker in ["_apply_upr_cre_score", "_prototype_score_for_rows", "_row_confidence"]:
    assert marker in wsl_text, f"Missing {marker} in wsl.py"
assert "--csv-output" in collector_text, "collect_relation_stats.py must accept --csv-output"

subprocess.run(["python", "-m", "py_compile", "main.py", "wsl.py", "relation_metrics.py", "scripts/collect_relation_stats.py"], cwd=wsl_dir, check=True)
print("Static checks OK.")


Static checks OK.


## Block 4 — Locate or extract phase1 checkpoint

This is the only non-parallel part. It trains phase 1 once, saves `model_5.pth`, and uses that checkpoint for both phase-2 jobs.

Expected time on T4: roughly similar to your previous phase-1 run.

In [5]:
from pathlib import Path
import tarfile
import shutil

work_dir = Path(CFG["WORK_DIR"])
standard_ckpt_dir = work_dir / "step5b_checkpoint"
standard_ckpt_dir.mkdir(parents=True, exist_ok=True)
standard_ckpt = standard_ckpt_dir / "phase1_model_5.pth"

candidates = []

# 1) Explicit path
if CFG["PHASE1_CKPT_PATH"]:
    p = Path(CFG["PHASE1_CKPT_PATH"])
    if p.exists():
        candidates.append(p)
    else:
        print("Explicit PHASE1_CKPT_PATH does not exist:", p)

# 2) Common working path
common = Path("/kaggle/working/artifacts_step5_fresh_t4x2/checkpoints/phase1_model_5.pth")
if common.exists():
    candidates.append(common)

# 3) Search input and working for phase1_model_5.pth/model_5.pth
for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    if root.exists():
        candidates.extend(root.rglob("phase1_model_5.pth"))
        # fallback: any model_5.pth under likely folders
        candidates.extend([p for p in root.rglob("model_5.pth") if "saved_pretrain" in str(p) or "phase1" in str(p)])

# 4) If only tar.gz exists, extract checkpoint from it
if not candidates:
    tar_candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            tar_candidates.extend(root.rglob("artifacts_step5_fresh_t4x2.tar.gz"))
            tar_candidates.extend(root.rglob("*.tar.gz"))
    for tar_path in tar_candidates:
        try:
            with tarfile.open(tar_path, "r:gz") as tf:
                members = [m for m in tf.getmembers() if m.name.endswith("phase1_model_5.pth") or m.name.endswith("model_5.pth")]
                if members:
                    print("Extracting checkpoint from", tar_path)
                    member = members[0]
                    extracted = tf.extractfile(member)
                    if extracted is None:
                        continue
                    with standard_ckpt.open("wb") as f:
                        shutil.copyfileobj(extracted, f)
                    candidates.append(standard_ckpt)
                    break
        except Exception as exc:
            print("Skip tar", tar_path, repr(exc))

# Remove duplicates while preserving order
seen = set()
uniq = []
for p in candidates:
    rp = str(p.resolve()) if p.exists() else str(p)
    if rp not in seen and p.exists():
        seen.add(rp)
        uniq.append(p)

if not uniq:
    raise FileNotFoundError(
        "No phase1 checkpoint found. Add artifacts_step5_fresh_t4x2 as Kaggle input, "
        "or set CFG['PHASE1_CKPT_PATH'] to the checkpoint path."
    )

source_ckpt = uniq[0]
if source_ckpt.resolve() != standard_ckpt.resolve():
    shutil.copy2(source_ckpt, standard_ckpt)

PHASE1_CKPT = str(standard_ckpt)
print("Using PHASE1_CKPT =", PHASE1_CKPT)
print("Checkpoint size MiB =", standard_ckpt.stat().st_size / (1024**2))


Using PHASE1_CKPT = /kaggle/working/step5b_checkpoint/phase1_model_5.pth
Checkpoint size MiB = 94.88339710235596


## Block 5 — Launch two parallel Step 5b runs on T4x2

Run this after Block 4. It finds the `model_5.pth` created by the fresh phase-1 run.

In [6]:
import subprocess
import os
from pathlib import Path
import json

repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"

phase1_ckpt = Path(CFG["PHASE1_CKPT_PATH"])
assert phase1_ckpt.exists(), f"Missing checkpoint: {phase1_ckpt}"

run_logs_dir = Path("/kaggle/working/run_logs")
run_logs_dir.mkdir(parents=True, exist_ok=True)

saved_root = wsl_dir.parent / "saved_regdb_resnet"

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=wsl_dir
).decode().strip()

print("Commit:", commit)
print("Phase1 checkpoint:", phase1_ckpt)

def build_cmd(run_tag, enable_upr, beta, gamma, warmup, device_id):
    save_path = f"{run_tag}_{RUN_SUFFIX}_{commit}"
    relation_stats_dir = f"../saved_regdb_resnet/{save_path}_1/relation_stats"

    cmd = [
        "python", "main.py",
        "--dataset", "regdb",
        "--data-path", CFG["DATA_ROOT"],
        "--debug", "wsl",
        "--save-path", save_path,
        "--arch", "resnet",
        "--trial", str(CFG["TRIAL"]),
        "--seed", str(CFG["SEED"]),
        "--model-path", str(phase1_ckpt),
        "--stage1-epoch", "0",
        "--stage2-epoch", str(CFG["STAGE2_EPOCH"]),
        "--lr", str(CFG["LR"]),
        "--batch-pidnum", str(CFG["BATCH_PIDNUM"]),
        "--pid-numsample", str(CFG["PID_NUMSAMPLE"]),
        "--test-batch", str(CFG["TEST_BATCH"]),
        "--num-workers", str(CFG["NUM_WORKERS"]),
        "--device", "0",
        "--milestones", *CFG["MILESTONES"].split(),
        "--save-relation-stats",
        "--relation-stats-dir", relation_stats_dir,
    ]

    if enable_upr:
        cmd += [
            "--upr-cre",
            "--upr-beta", str(beta),
            "--upr-gamma", str(gamma),
            "--upr-margin-weight", "1.0",
            "--upr-warmup-epoch", str(warmup),
        ]

    log_path = run_logs_dir / f"{save_path}.log"

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(device_id)

    return cmd, log_path, env, save_path, relation_stats_dir

jobs = []

configs = [
    {
        "tag": CFG["RUN_A_TAG"],
        "enable_upr": CFG["RUN_A_ENABLE_UPR"],
        "beta": CFG["RUN_A_BETA"],
        "gamma": CFG["RUN_A_GAMMA"],
        "warmup": CFG["RUN_A_WARMUP"],
        "gpu": 0,
    },
    {
        "tag": CFG["RUN_B_TAG"],
        "enable_upr": CFG["RUN_B_ENABLE_UPR"],
        "beta": CFG["RUN_B_BETA"],
        "gamma": CFG["RUN_B_GAMMA"],
        "warmup": CFG["RUN_B_WARMUP"],
        "gpu": 1,
    },
]

for cfg in configs:
    cmd, log_path, env, save_path, relation_stats_dir = build_cmd(
        run_tag=cfg["tag"],
        enable_upr=cfg["enable_upr"],
        beta=cfg["beta"],
        gamma=cfg["gamma"],
        warmup=cfg["warmup"],
        device_id=cfg["gpu"],
    )

    print("\nStarting:", cfg["tag"])
    print("GPU:", cfg["gpu"])
    print("Enable UPR:", cfg["enable_upr"])
    print("Save path:", save_path)
    print("Log:", log_path)
    print("Command:", " ".join(cmd))

    f = open(log_path, "w")
    proc = subprocess.Popen(
        cmd,
        cwd=wsl_dir,
        stdout=f,
        stderr=subprocess.STDOUT,
        env=env,
    )

    jobs.append({
        "tag": cfg["tag"],
        "pid": proc.pid,
        "process": proc,
        "log_path": str(log_path),
        "save_path": save_path,
        "relation_stats_dir": relation_stats_dir,
        "enable_upr": cfg["enable_upr"],
        "beta": cfg["beta"],
        "gamma": cfg["gamma"],
        "warmup": cfg["warmup"],
    })

print("\nStarted jobs:")
for job in jobs:
    print(job["tag"], "PID:", job["pid"], "log:", job["log_path"])

runtime_info = {
    "run_suffix": RUN_SUFFIX,
    "commit": commit,
    "phase1_ckpt": str(phase1_ckpt),
    "jobs": [
        {k: v for k, v in job.items() if k != "process"}
        for job in jobs
    ],
}

Path("/kaggle/working/step6_runtime_info.json").write_text(
    json.dumps(runtime_info, indent=2)
)

print("\nSaved runtime info to /kaggle/working/step6_runtime_info.json")

Commit: 6c96804
Phase1 checkpoint: /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth

Starting: baseline_p2s15
GPU: 0
Enable UPR: False
Save path: baseline_p2s15_step6_20260623_063755_6c96804
Log: /kaggle/working/run_logs/baseline_p2s15_step6_20260623_063755_6c96804.log
Command: python main.py --dataset regdb --data-path /kaggle/working/VIREID_Dataset --debug wsl --save-path baseline_p2s15_step6_20260623_063755_6c96804 --arch resnet --trial 1 --seed 1 --model-path /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth --stage1-epoch 0 --stage2-epoch 15 --lr 0.00045 --batch-pidnum 5 --pid-numsample 4 --test-batch 64 --num-workers 2 --device 0 --milestones 8 12 --save-relation-stats --relation-stats-dir ../saved_regdb_resnet/baseline_p2s15_step6_20260623_063755_6c96804_1/relation_stats

Starting: upr_b01_g00_w2_p2s15
GPU: 1
Enable UPR: True
Save path: upr_b01_g00_w2_p2s15_step6_20260623_063755_6c96804
Log: /kaggle/working/run_logs/up

## Block 6 - Monitor logs and GPUs. Re-run this cell while jobs are running. long validation for best v0.1 config

GPU 0: baseline phase2-only, 5 epochs.  
GPU 1: UPR-CRE v0.1 phase2-only, `beta=0.1`, `gamma=0.5`, `warmup=2`, 5 epochs.

Both jobs use the same phase-1 checkpoint from Block 5.

In [7]:
import subprocess
import time
from pathlib import Path
import json

info = json.loads(Path("/kaggle/working/step6_runtime_info.json").read_text())

print("=== nvidia-smi ===")
subprocess.run(["nvidia-smi"])

for job in info["jobs"]:
    print("\n==============================")
    print(job["tag"])
    print("log:", job["log_path"])
    print("==============================")
    log_path = Path(job["log_path"])
    if log_path.exists():
        subprocess.run(["tail", "-40", str(log_path)])
    else:
        print("Log not found yet.")

=== nvidia-smi ===
Tue Jun 23 06:38:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

## Block 7 — Monitor jobs

Run this cell repeatedly while the two jobs are running.

In [8]:
import time
from pathlib import Path
import json
import subprocess

info = json.loads(Path("/kaggle/working/step6_runtime_info.json").read_text())

# Nếu jobs object còn nằm trong memory từ block chạy song song thì dùng trực tiếp.
# Nếu không còn, chỉ monitor bằng log/process thủ công.
if "jobs" in globals():
    for job in jobs:
        ret = job["process"].wait()
        print(job["tag"], "finished with code", ret)
else:
    print("jobs variable not found. Check processes manually:")
    subprocess.run("ps -ef | grep 'python main.py' | grep -v grep", shell=True)

baseline_p2s15 finished with code 0
upr_b01_g00_w2_p2s15 finished with code 0


## Block 8 - Collect metrics, relation stats, and comparison CSV

This polls the two background PIDs from Block 6. You can stop this cell if you only want manual monitoring.

In [9]:
from pathlib import Path
import csv
import json
import re
import subprocess
import shutil

repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"

info = json.loads(Path("/kaggle/working/step6_runtime_info.json").read_text())

rows = []

def parse_best_metrics(log_text):
    pattern = re.compile(
        r"R1:([0-9.]+);\s+R10:([0-9.]+);\s+R20:([0-9.]+);\s+mAP:([0-9.]+);\s+mINP:([0-9.]+)"
    )
    matches = pattern.findall(log_text)
    if not matches:
        return "", "", "", "", ""

    # Chọn best theo mAP
    best = max(matches, key=lambda x: float(x[3]))
    r1, r10, r20, map_, minp = best
    return r1, r10, r20, map_, minp

for job in info["jobs"]:
    run_dir = wsl_dir.parent / "saved_regdb_resnet" / f"{job['save_path']}_1"
    log_path = run_dir / "log" / "log.txt"
    stats_dir = run_dir / "relation_stats"
    stats_csv = stats_dir / "relation_stats_summary.csv"

    print("Collecting:", job["tag"])
    print("run_dir:", run_dir)

    if stats_dir.exists():
        subprocess.run([
            "python", "scripts/collect_relation_stats.py",
            "--stats-dir", str(stats_dir),
            "--csv-output", str(stats_csv),
        ], cwd=wsl_dir, check=True)

    log_text = log_path.read_text() if log_path.exists() else ""
    r1, r10, r20, map_, minp = parse_best_metrics(log_text)

    last_stats = {}
    if stats_csv.exists():
        with stats_csv.open() as f:
            reader = list(csv.DictReader(f))
            if reader:
                last_stats = reader[-1]

    rows.append({
        "run": job["tag"],
        "save_path": job["save_path"],
        "enable_upr": job["enable_upr"],
        "beta": job["beta"],
        "gamma": job["gamma"],
        "warmup": job["warmup"],
        "stage2_epoch": CFG["STAGE2_EPOCH"],
        "best_r1_by_map": r1,
        "best_map": map_,
        "best_minp": minp,
        "last_common_accuracy": last_stats.get("common_accuracy", ""),
        "last_mutual_match_ratio": last_stats.get("mutual_match_ratio", ""),
        "last_num_common": last_stats.get("num_common", ""),
        "last_num_specific": last_stats.get("num_specific", ""),
        "last_num_remain": last_stats.get("num_remain", ""),
    })

out_csv = Path("/kaggle/working/step6_p2s15_baseline_vs_upr_summary.csv")
with out_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("\nSummary:")
print(out_csv.read_text())
print("Saved:", out_csv)

Collecting: baseline_p2s15
run_dir: /kaggle/working/UIT.CS2309/saved_regdb_resnet/baseline_p2s15_step6_20260623_063755_6c96804_1
Wrote /kaggle/working/UIT.CS2309/saved_regdb_resnet/baseline_p2s15_step6_20260623_063755_6c96804_1/relation_stats/relation_stats_summary.csv
{'epoch': 0, 'num_r2i_pairs': 142, 'num_i2r_pairs': 124, 'num_common': 47, 'num_specific': 32, 'num_remain': 99, 'mutual_match_ratio': 0.33098591549295775, 'r2i_pair_accuracy': 0.3732394366197183, 'i2r_pair_accuracy': 0.3467741935483871, 'common_accuracy': 0.5957446808510638, 'rgb_entropy_mean': 0.17003308491088964, 'ir_entropy_mean': 0.18085551654863424, 'rgb_margin_mean': 0.5666432981771526, 'ir_margin_mean': 0.5349934934331859, 'proto_common_mean': 0.4523059738443253, 'proto_specific_mean': 0.2990252198651433, 'proto_remain_mean': 0.3325949123110434}
{'epoch': 1, 'num_r2i_pairs': 151, 'num_i2r_pairs': 132, 'num_common': 59, 'num_specific': 32, 'num_remain': 95, 'mutual_match_ratio': 0.39072847682119205, 'r2i_pair_accu

## Block 9 - Archive Step 5b artifacts

This creates `/kaggle/working/step5_fresh_t4x2_summary.csv`.

In [10]:
from pathlib import Path
import json
import shutil
import subprocess

repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"

info = json.loads(Path("/kaggle/working/step6_runtime_info.json").read_text())

artifact_dir = Path("/kaggle/working/artifacts_step6_p2s15")
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)

(artifact_dir / "logs").mkdir(parents=True, exist_ok=True)
(artifact_dir / "relation_stats").mkdir(parents=True, exist_ok=True)

# Copy summary and runtime info
for p in [
    Path("/kaggle/working/step6_runtime_info.json"),
    Path("/kaggle/working/step6_p2s15_baseline_vs_upr_summary.csv"),
]:
    if p.exists():
        shutil.copy2(p, artifact_dir / p.name)

# Copy logs and relation stats
for job in info["jobs"]:
    run_dir = wsl_dir.parent / "saved_regdb_resnet" / f"{job['save_path']}_1"

    log_file = run_dir / "log" / "log.txt"
    if log_file.exists():
        shutil.copy2(log_file, artifact_dir / "logs" / f"{job['save_path']}_log.txt")

    stats_csv = run_dir / "relation_stats" / "relation_stats_summary.csv"
    if stats_csv.exists():
        shutil.copy2(
            stats_csv,
            artifact_dir / "relation_stats" / f"{job['save_path']}_relation_stats_summary.csv"
        )

tar_path = Path("/kaggle/working/artifacts_step6_p2s15.tar.gz")
if tar_path.exists():
    tar_path.unlink()

subprocess.run([
    "tar", "-czf", str(tar_path),
    "-C", "/kaggle/working",
    artifact_dir.name
], check=True)

print("Artifact:", tar_path)
print("Size MB:", tar_path.stat().st_size / (1024 * 1024))

Artifact: /kaggle/working/artifacts_step6_p2s15.tar.gz
Size MB: 0.0074825286865234375


## Block 10 - Optional: upload final artifact to GitHub Release
This is disabled by default. Use it only if you want persistence outside Kaggle.
It uploads /kaggle/working/artifacts_step5b_t4x2.tar.gz as a release asset.

In [12]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import os
import json
import shutil
import subprocess
import textwrap
import hashlib
from datetime import datetime

# =========================
# CONFIG
# =========================
OWNER = CFG.get("GITHUB_OWNER", "TranTruongMMCII")
REPO_NAME = CFG.get("REPO_NAME", "UIT.CS2309")
BRANCH = CFG.get("BRANCH", "feature/upr-cre")
WORK_DIR = Path(CFG.get("WORK_DIR", "/kaggle/working"))
REPO_DIR = WORK_DIR / REPO_NAME

GITHUB_TOKEN_SECRET = CFG.get("GITHUB_TOKEN_SECRET", "GITHUB_TOKEN")
GIT_USER_NAME = CFG.get("GIT_USER_NAME", "Kaggle Bot")
GIT_USER_EMAIL = CFG.get("GIT_USER_EMAIL", "kaggle-bot@example.com")

# Do not push large files into normal git by default.
INCLUDE_LARGE_FILES_IN_GIT = False
MAX_NORMAL_GIT_FILE_MB = 90

# Candidate artifact folders from recent notebooks.
ARTIFACT_CANDIDATES = [
    Path("/kaggle/working/artifacts_step5b_t4x2"),
    Path("/kaggle/working/artifacts_step5_fresh_t4x2"),
    Path("/kaggle/working/artifacts_step4_v01_partial"),
]

TAR_CANDIDATES = [
    Path("/kaggle/working/artifacts_step5b_t4x2.tar.gz"),
    Path("/kaggle/working/artifacts_step5_fresh_t4x2.tar.gz"),
    Path("/kaggle/working/artifacts_step4_v01_partial.tar.gz"),
]

# =========================
# HELPERS
# =========================
def run(cmd, cwd=REPO_DIR, env=None, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def copy_if_exists(src: Path, dst: Path):
    if not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    return True

# =========================
# 1. Resolve artifact source
# =========================
artifact_dir = next((p for p in ARTIFACT_CANDIDATES if p.exists()), None)
artifact_tar = next((p for p in TAR_CANDIDATES if p.exists()), None)

if artifact_dir is None and artifact_tar is None:
    raise FileNotFoundError(
        "No artifact folder/tar found. Expected one of:\n"
        + "\n".join(str(p) for p in ARTIFACT_CANDIDATES + TAR_CANDIDATES)
    )

runtime_info_path = Path("/kaggle/working/step5b_runtime_info.json")
if runtime_info_path.exists():
    runtime_info = json.loads(runtime_info_path.read_text())
    run_id = runtime_info.get("run_suffix", "")
else:
    commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR,
    ).decode().strip()
    run_id = f"manual_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{commit}"
    runtime_info = {"run_suffix": run_id, "commit": commit}

if not run_id:
    run_id = f"manual_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

backup_dir = REPO_DIR / "experiments" / "kaggle_runs" / run_id
backup_dir.mkdir(parents=True, exist_ok=True)

print("Artifact dir:", artifact_dir)
print("Artifact tar:", artifact_tar)
print("Backup dir:", backup_dir)

# =========================
# 2. Copy small artifacts
# =========================
# Runtime info
(backup_dir / "runtime_info.json").write_text(
    json.dumps(runtime_info, indent=2, ensure_ascii=False)
)

# Copy summary CSV files from /kaggle/working
for csv_path in Path("/kaggle/working").glob("*summary*.csv"):
    copy_if_exists(csv_path, backup_dir / csv_path.name)

# Copy logs and relation stats if artifact folder exists
if artifact_dir is not None:
    for subdir in ["logs", "run_logs", "relation_stats"]:
        src = artifact_dir / subdir
        if src.exists():
            copy_if_exists(src, backup_dir / subdir)

    # Copy top-level txt/csv/json files
    for pattern in ["*.csv", "*.txt", "*.json"]:
        for src in artifact_dir.glob(pattern):
            copy_if_exists(src, backup_dir / src.name)

# Copy run_logs from global working folder
global_run_logs = Path("/kaggle/working/run_logs")
if global_run_logs.exists():
    copy_if_exists(global_run_logs, backup_dir / "run_logs")

# =========================
# 3. Record large files in manifest
# =========================
large_files = []

if artifact_tar is not None:
    large_files.append(artifact_tar)

if artifact_dir is not None:
    for p in artifact_dir.rglob("*"):
        if p.is_file() and p.suffix in [".pth", ".pt", ".ckpt", ".gz"]:
            large_files.append(p)

manifest = {
    "run_id": run_id,
    "repo": f"{OWNER}/{REPO_NAME}",
    "branch": BRANCH,
    "backup_dir_in_repo": str(backup_dir.relative_to(REPO_DIR)),
    "artifact_dir": str(artifact_dir) if artifact_dir else None,
    "artifact_tar": str(artifact_tar) if artifact_tar else None,
    "large_files": [],
    "note": (
        "Large files are not committed by default. "
        "Upload .pth/.tar.gz to Kaggle Dataset/Input if you need them across sessions."
    ),
}

large_dst_dir = backup_dir / "large_files"
large_dst_dir.mkdir(parents=True, exist_ok=True)

for src in sorted(set(large_files)):
    size_mb = src.stat().st_size / (1024 * 1024)
    item = {
        "source_path": str(src),
        "name": src.name,
        "size_mb": round(size_mb, 2),
        "sha256": sha256_file(src) if size_mb <= 1024 else "skipped_for_large_file",
        "committed_to_git": False,
    }

    if INCLUDE_LARGE_FILES_IN_GIT and size_mb <= MAX_NORMAL_GIT_FILE_MB:
        dst = large_dst_dir / src.name
        shutil.copy2(src, dst)
        item["committed_to_git"] = True
        item["repo_path"] = str(dst.relative_to(REPO_DIR))
    elif INCLUDE_LARGE_FILES_IN_GIT and size_mb > MAX_NORMAL_GIT_FILE_MB:
        item["skip_reason"] = f"larger than {MAX_NORMAL_GIT_FILE_MB} MB"

    manifest["large_files"].append(item)

(backup_dir / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False)
)

# README
(backup_dir / "README.md").write_text(f"""# Kaggle run backup: {run_id}

This directory was pushed from Kaggle using normal `git push`.

## Contents

- Small logs, CSV summaries, relation statistics, and manifest files.
- Large files such as `.pth` or `.tar.gz` are recorded in `manifest.json`.

## Large artifact policy

By default, checkpoints and full `.tar.gz` artifacts are **not committed** into git.
Upload those files to Kaggle Dataset/Input if they are needed across independent Kaggle sessions.

""")

print("Prepared backup directory:")
for p in sorted(backup_dir.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(REPO_DIR), f"({p.stat().st_size/1024:.1f} KiB)")

# =========================
# 4. Git push using old GIT_ASKPASS method
# =========================
token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET).strip()

askpass = WORK_DIR / "git_askpass_push.sh"
askpass.write_text(textwrap.dedent(f"""\
#!/bin/sh
case "$1" in
  *Username*) echo "{OWNER}" ;;
  *Password*) echo "{token}" ;;
  *) echo "" ;;
esac
"""))
askpass.chmod(0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

public_url = f"https://github.com/{OWNER}/{REPO_NAME}.git"

try:
    run(["git", "config", "user.name", GIT_USER_NAME], env=env)
    run(["git", "config", "user.email", GIT_USER_EMAIL], env=env)
    run(["git", "remote", "set-url", "origin", public_url], env=env)

    # Sync branch first.
    run(["git", "fetch", "origin", BRANCH], env=env)
    run(["git", "checkout", BRANCH], env=env)
    run(["git", "pull", "--rebase", "origin", BRANCH], env=env)

    # Force add because some repos ignore logs/ or *.log.
    rel_backup = backup_dir.relative_to(REPO_DIR)
    run(["git", "add", "-f", str(rel_backup)], env=env)

    # Commit only if there are staged changes.
    diff_status = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_DIR,
        env=env,
    )

    if diff_status.returncode == 0:
        print("No staged changes to commit.")
    else:
        run(["git", "commit", "-m", f"exp: add Kaggle run backup {run_id}"], env=env)
        run(["git", "push", "origin", BRANCH], env=env)

    print("Backup pushed to:")
    print(f"https://github.com/{OWNER}/{REPO_NAME}/tree/{BRANCH}/experiments/kaggle_runs/{run_id}")

finally:
    # Remove askpass file so token is not left in /kaggle/working.
    try:
        askpass.unlink()
    except FileNotFoundError:
        pass

FileNotFoundError: No artifact folder/tar found. Expected one of:
/kaggle/working/artifacts_step5b_t4x2
/kaggle/working/artifacts_step5_fresh_t4x2
/kaggle/working/artifacts_step4_v01_partial
/kaggle/working/artifacts_step5b_t4x2.tar.gz
/kaggle/working/artifacts_step5_fresh_t4x2.tar.gz
/kaggle/working/artifacts_step4_v01_partial.tar.gz

In [13]:
from pathlib import Path
import json
import shutil
import subprocess

WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "UIT.CS2309"
WSL_DIR = REPO_DIR / "WSL_ReID"

runtime_path = WORK_DIR / "step6_runtime_info.json"
summary_path = WORK_DIR / "step6_p2s15_baseline_vs_upr_summary.csv"

assert runtime_path.exists(), f"Missing {runtime_path}"
assert summary_path.exists(), f"Missing {summary_path}"

info = json.loads(runtime_path.read_text())

artifact_dir = WORK_DIR / "artifacts_step6_p2s15"
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)

(artifact_dir / "logs").mkdir(parents=True, exist_ok=True)
(artifact_dir / "relation_stats").mkdir(parents=True, exist_ok=True)

shutil.copy2(runtime_path, artifact_dir / runtime_path.name)
shutil.copy2(summary_path, artifact_dir / summary_path.name)

for job in info["jobs"]:
    run_dir = WSL_DIR.parent / "saved_regdb_resnet" / f"{job['save_path']}_1"

    log_file = run_dir / "log" / "log.txt"
    if log_file.exists():
        shutil.copy2(log_file, artifact_dir / "logs" / f"{job['save_path']}_log.txt")

    stats_csv = run_dir / "relation_stats" / "relation_stats_summary.csv"
    if stats_csv.exists():
        shutil.copy2(
            stats_csv,
            artifact_dir / "relation_stats" / f"{job['save_path']}_relation_stats_summary.csv"
        )

tar_path = WORK_DIR / "artifacts_step6_p2s15.tar.gz"
if tar_path.exists():
    tar_path.unlink()

subprocess.run([
    "tar", "-czf", str(tar_path),
    "-C", str(WORK_DIR),
    artifact_dir.name,
], check=True)

print("Created:", artifact_dir)
print("Created:", tar_path)
print("Size MB:", tar_path.stat().st_size / (1024 * 1024))

Created: /kaggle/working/artifacts_step6_p2s15
Created: /kaggle/working/artifacts_step6_p2s15.tar.gz
Size MB: 0.0074863433837890625


In [14]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import os
import json
import shutil
import subprocess
import textwrap
import hashlib
from datetime import datetime

# =========================
# CONFIG
# =========================
OWNER = CFG.get("GITHUB_OWNER", "TranTruongMMCII")
REPO_NAME = CFG.get("REPO_NAME", "UIT.CS2309")
BRANCH = CFG.get("BRANCH", "feature/upr-cre")
WORK_DIR = Path(CFG.get("WORK_DIR", "/kaggle/working"))
REPO_DIR = WORK_DIR / REPO_NAME

GITHUB_TOKEN_SECRET = CFG.get("GITHUB_TOKEN_SECRET", "GITHUB_TOKEN")
GIT_USER_NAME = CFG.get("GIT_USER_NAME", "Kaggle Bot")
GIT_USER_EMAIL = CFG.get("GIT_USER_EMAIL", "kaggle-bot@example.com")

# Step 6 artifact paths
ARTIFACT_DIR = WORK_DIR / "artifacts_step6_p2s15"
ARTIFACT_TAR = WORK_DIR / "artifacts_step6_p2s15.tar.gz"
RUNTIME_INFO = WORK_DIR / "step6_runtime_info.json"
SUMMARY_CSV = WORK_DIR / "step6_p2s15_baseline_vs_upr_summary.csv"

# Do not push large files into normal git by default.
INCLUDE_LARGE_FILES_IN_GIT = False
MAX_NORMAL_GIT_FILE_MB = 90

# =========================
# HELPERS
# =========================
def run(cmd, cwd=REPO_DIR, env=None, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def copy_if_exists(src: Path, dst: Path):
    if not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    return True

# =========================
# 1. Validate Step 6 outputs
# =========================
if not ARTIFACT_DIR.exists() and not ARTIFACT_TAR.exists():
    raise FileNotFoundError(
        "Missing Step 6 artifact. Expected one of:\n"
        f"- {ARTIFACT_DIR}\n"
        f"- {ARTIFACT_TAR}\n"
        "Run the Step 6 archive block first."
    )

if not RUNTIME_INFO.exists():
    raise FileNotFoundError(f"Missing runtime info: {RUNTIME_INFO}")

runtime_info = json.loads(RUNTIME_INFO.read_text())
run_id = runtime_info.get("run_suffix", "")

if not run_id:
    commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR,
    ).decode().strip()
    run_id = f"step6_manual_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{commit}"
    runtime_info["run_suffix"] = run_id

print("Run ID:", run_id)
print("Artifact dir:", ARTIFACT_DIR if ARTIFACT_DIR.exists() else None)
print("Artifact tar:", ARTIFACT_TAR if ARTIFACT_TAR.exists() else None)

# =========================
# 2. Prepare git auth first
# =========================
token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET).strip()

askpass = WORK_DIR / "git_askpass_push.sh"
askpass.write_text(textwrap.dedent(f"""\
#!/bin/sh
case "$1" in
  *Username*) echo "{OWNER}" ;;
  *Password*) echo "{token}" ;;
  *) echo "" ;;
esac
"""))
askpass.chmod(0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

public_url = f"https://github.com/{OWNER}/{REPO_NAME}.git"

try:
    # =========================
    # 3. Sync branch before copying backup
    # =========================
    run(["git", "config", "user.name", GIT_USER_NAME], env=env)
    run(["git", "config", "user.email", GIT_USER_EMAIL], env=env)
    run(["git", "remote", "set-url", "origin", public_url], env=env)

    run(["git", "fetch", "origin", BRANCH], env=env)
    run(["git", "checkout", BRANCH], env=env)
    run(["git", "pull", "--rebase", "origin", BRANCH], env=env)

    # =========================
    # 4. Copy small Step 6 artifacts into repo
    # =========================
    backup_dir = REPO_DIR / "experiments" / "kaggle_runs" / run_id
    if backup_dir.exists():
        shutil.rmtree(backup_dir)
    backup_dir.mkdir(parents=True, exist_ok=True)

    (backup_dir / "runtime_info.json").write_text(
        json.dumps(runtime_info, indent=2, ensure_ascii=False)
    )

    copy_if_exists(SUMMARY_CSV, backup_dir / SUMMARY_CSV.name)

    if ARTIFACT_DIR.exists():
        for subdir in ["logs", "run_logs", "relation_stats"]:
            src = ARTIFACT_DIR / subdir
            if src.exists():
                copy_if_exists(src, backup_dir / subdir)

        for pattern in ["*.csv", "*.txt", "*.json"]:
            for src in ARTIFACT_DIR.glob(pattern):
                copy_if_exists(src, backup_dir / src.name)

    global_run_logs = WORK_DIR / "run_logs"
    if global_run_logs.exists():
        copy_if_exists(global_run_logs, backup_dir / "run_logs")

    # =========================
    # 5. Manifest for large files
    # =========================
    large_files = []

    if ARTIFACT_TAR.exists():
        large_files.append(ARTIFACT_TAR)

    if ARTIFACT_DIR.exists():
        for p in ARTIFACT_DIR.rglob("*"):
            if p.is_file() and p.suffix in [".pth", ".pt", ".ckpt", ".gz"]:
                large_files.append(p)

    manifest = {
        "run_id": run_id,
        "repo": f"{OWNER}/{REPO_NAME}",
        "branch": BRANCH,
        "backup_dir_in_repo": str(backup_dir.relative_to(REPO_DIR)),
        "artifact_dir": str(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None,
        "artifact_tar": str(ARTIFACT_TAR) if ARTIFACT_TAR.exists() else None,
        "large_files": [],
        "note": (
            "Large files are not committed by default. "
            "Upload .pth/.tar.gz to Kaggle Dataset/Input if needed across sessions."
        ),
    }

    large_dst_dir = backup_dir / "large_files"
    large_dst_dir.mkdir(parents=True, exist_ok=True)

    for src in sorted(set(large_files)):
        size_mb = src.stat().st_size / (1024 * 1024)
        item = {
            "source_path": str(src),
            "name": src.name,
            "size_mb": round(size_mb, 2),
            "sha256": sha256_file(src) if size_mb <= 1024 else "skipped_for_large_file",
            "committed_to_git": False,
        }

        if INCLUDE_LARGE_FILES_IN_GIT and size_mb <= MAX_NORMAL_GIT_FILE_MB:
            dst = large_dst_dir / src.name
            shutil.copy2(src, dst)
            item["committed_to_git"] = True
            item["repo_path"] = str(dst.relative_to(REPO_DIR))
        elif INCLUDE_LARGE_FILES_IN_GIT and size_mb > MAX_NORMAL_GIT_FILE_MB:
            item["skip_reason"] = f"larger than {MAX_NORMAL_GIT_FILE_MB} MB"

        manifest["large_files"].append(item)

    (backup_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )

    (backup_dir / "README.md").write_text(f"""# Kaggle run backup: {run_id}

This directory was pushed from Kaggle using normal `git push`.

## Contents

- Small logs, CSV summaries, relation statistics, and manifest files.
- Large files such as `.pth` or `.tar.gz` are recorded in `manifest.json`.

## Large artifact policy

By default, checkpoints and full `.tar.gz` artifacts are **not committed** into git.
Upload those files to Kaggle Dataset/Input if they are needed across independent Kaggle sessions.

""")

    print("Prepared backup directory:")
    for p in sorted(backup_dir.rglob("*")):
        if p.is_file():
            print(" -", p.relative_to(REPO_DIR), f"({p.stat().st_size/1024:.1f} KiB)")

    # =========================
    # 6. Commit and push
    # =========================
    rel_backup = backup_dir.relative_to(REPO_DIR)
    run(["git", "add", "-f", str(rel_backup)], env=env)

    diff_status = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_DIR,
        env=env,
    )

    if diff_status.returncode == 0:
        print("No staged changes to commit.")
    else:
        run(["git", "commit", "-m", f"exp: add Step 6 Kaggle run backup {run_id}"], env=env)
        run(["git", "push", "origin", BRANCH], env=env)

    print("Backup pushed to:")
    print(f"https://github.com/{OWNER}/{REPO_NAME}/tree/{BRANCH}/experiments/kaggle_runs/{run_id}")

finally:
    try:
        askpass.unlink()
    except FileNotFoundError:
        pass

Run ID: step6_20260623_063755
Artifact dir: /kaggle/working/artifacts_step6_p2s15
Artifact tar: /kaggle/working/artifacts_step6_p2s15.tar.gz
+ git config user.name Kaggle Bot
+ git config user.email kaggle-bot@example.com
+ git remote set-url origin https://github.com/TranTruongMMCII/UIT.CS2309.git
+ git fetch origin feature/upr-cre


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD
Already on 'feature/upr-cre'
From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD


+ git checkout feature/upr-cre
Your branch is up to date with 'origin/feature/upr-cre'.
+ git pull --rebase origin feature/upr-cre
Already up to date.
Prepared backup directory:
 - experiments/kaggle_runs/step6_20260623_063755/README.md (0.5 KiB)
 - experiments/kaggle_runs/step6_20260623_063755/logs/baseline_p2s15_step6_20260623_063755_6c96804_log.txt (11.2 KiB)
 - experiments/kaggle_runs/step6_20260623_063755/logs/upr_b01_g00_w2_p2s15_step6_20260623_063755_6c96804_log.txt (11.3 KiB)
 - experiments/kaggle_runs/step6_20260623_063755/manifest.json (0.7 KiB)
 - experiments/kaggle_runs/step6_20260623_063755/relation_stats/baseline_p2s15_step6_20260623_063755_6c96804_relation_stats_summary.csv (3.6 KiB)
 - experiments/kaggle_runs/step6_20260623_063755/relation_stats/upr_b01_g00_w2_p2s15_step6_20260623_063755_6c96804_relation_stats_summary.csv (3.6 KiB)
 - experiments/kaggle_runs/step6_20260623_063755/run_logs/baseline_p2s15_step6_20260623_063755_6c96804.log (13.1 KiB)
 - experiments/kaggle_

remote: 
remote: GitHub found 12 vulnerabilities on TranTruongMMCII/UIT.CS2309's default branch (1 critical, 2 high, 3 moderate, 6 low). To find out more, visit:        
remote:      https://github.com/TranTruongMMCII/UIT.CS2309/security/dependabot        
remote: 
To https://github.com/TranTruongMMCII/UIT.CS2309.git
   6c96804..b7db73a  feature/upr-cre -> feature/upr-cre
